In [1]:

# =========================
# Khai báo thư viện
# =========================
from pathlib import Path
from matplotlib.ticker import PercentFormatter
import json
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# try:
#     from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
# except ImportError:
#     subprocess.check_call([
#         sys.executable,
#         "-m",
#         "pip",
#         "install",
#         "iterative-stratification",
#         "-q",
#     ])
#     from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
# =========================
# Đường dẫn cố định
# =========================

ROOT_DIRECTORY   = (Path.cwd().parent).resolve()
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff",}

class TqdmComma(tqdm):

    @staticmethod
    def format_num(n):
        if isinstance(n, int) or float(n).is_integer():
            return f"{int(n):,}"

        return f"{n:,.2f}"

In [2]:
IMAGE_DIRECTORY             = (ROOT_DIRECTORY / "original_dataset" / "NIH_ChestXRay14" / "images").resolve()
DATA_ENTRY_CSV_DIRECTORY    = (ROOT_DIRECTORY / "original_dataset" / "NIH_ChestXRay14" / "Data_Entry_2017_v2020.csv").resolve()
TRAIN_SPLIT_DIRECTORY       = (ROOT_DIRECTORY / "original_dataset" / "NIH_ChestXRay14" / "train_val_list.txt").resolve()
TEST_SPLIT_DIRECTORY        = (ROOT_DIRECTORY / "original_dataset" / "NIH_ChestXRay14" / "test_list.txt").resolve()
OUTPUT_DIRECTORY            = (ROOT_DIRECTORY / "dataset").resolve()

RESIZE_IMAGES               = True
IMAGE_SIZE                  = 224

NUMBER_OF_FOLDS             = 5
RANDOM_SEED                 = 42

ID_COLUMN                   = "Image Index"
LABEL_COLUMN                = "Finding Labels"
PATIENT_ID_COLUMN           = "Patient ID"
NO_FINDING_LABEL            = "No Finding"
LABEL_SEPARATOR             = "|"    

In [3]:
MAX_TITLE_WIDTH = 30
MAX_MESSAGE_WIDTH = 20

all_valid = True

In [4]:
print("\n[1] KIỂM TRA ĐƯỜNG DẪN DỮ LIỆU ĐẦU VÀO")
print("-" * 80)

paths_to_check = {
    "IMAGE_DIRECTORY"          : IMAGE_DIRECTORY,
    "DATA_ENTRY_CSV_DIRECTORY" : DATA_ENTRY_CSV_DIRECTORY,
    "TRAIN_SPLIT_DIRECTORY"    : TRAIN_SPLIT_DIRECTORY,
    "TEST_SPLIT_DIRECTORY"     : TEST_SPLIT_DIRECTORY,
}

for name, path in paths_to_check.items():
    exists = path.exists()

    if exists:
        if path.is_dir():
            
            path_type = "Directory"
            
        elif path.is_file():
            
            path_type = "File"
            
        else:
            
            path_type = "Unknown"

        print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {name:<{MAX_TITLE_WIDTH}}: Type: {path_type:<10} : {path}")
        
    else:
        
        print(f"{'[MISSING]':<{MAX_MESSAGE_WIDTH}} {name:<{MAX_TITLE_WIDTH}}: {path}")
        all_valid = False

In [5]:
print("\n[2] KIỂM TRA CẤU HÌNH CỘT VÀ NHÃN")
print("-" * 80)

# =========================
# Kiểm tra cấu hình cột dữ liệu
# =========================
column_configs = {
    "ID_COLUMN"         : ID_COLUMN,
    "LABEL_COLUMN"      : LABEL_COLUMN,
    "PATIENT_ID_COLUMN" : PATIENT_ID_COLUMN,
    "LABEL_SEPARATOR"   : LABEL_SEPARATOR,
    "NO_FINDING_LABEL"  : NO_FINDING_LABEL,
}

for name, value in column_configs.items():

    if isinstance(value, str) and value.strip():

        print(f"{'[OK]':<{MAX_MESSAGE_WIDTH}} {name:<{MAX_TITLE_WIDTH}}: {value}"
        )

    else:

        print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {name:<{MAX_TITLE_WIDTH}}: Phải là chuỗi không rỗng.")
        all_valid = False

# =========================
# Kiểm tra tên cột trùng nhau
# =========================
required_column_names = [ID_COLUMN, LABEL_COLUMN, PATIENT_ID_COLUMN]

if len(required_column_names) == len(set(required_column_names)):

    print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'Kiểm tra cột trùng nhau':<{MAX_TITLE_WIDTH}}: Không trùng nhau")

else:

    print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {'Kiểm tra cột trùng nhau':<{MAX_TITLE_WIDTH}}: Có tên cột bị trùng")
    all_valid = False

In [6]:
print("\n[3] KIỂM TRA DỮ LIỆU TRONG TỆP TIN CSV")
print("-" * 80)

dataset_df = pd.read_csv(DATA_ENTRY_CSV_DIRECTORY)

required_columns = {ID_COLUMN, LABEL_COLUMN, PATIENT_ID_COLUMN}
loaded_dataframes = {}

dataset_checks = {
    "IMAGE ID THIẾU"  : int(dataset_df[ID_COLUMN].isna().sum()),
    "IMAGE ID TRÙNG"  : int(dataset_df[ID_COLUMN].duplicated().sum()),
    "PATIENT ID THIẾU": int(dataset_df[PATIENT_ID_COLUMN].isna().sum()),
    "LABEL THIẾU"     : int(dataset_df[LABEL_COLUMN].isna().sum()),
    "LABEL RỖNG"      : int(dataset_df[LABEL_COLUMN].fillna("").astype(str).str.strip().eq("").sum()),
}

for check_name, count in dataset_checks.items():
    status = "[OK]" if count == 0 else "[INVALID]"
    print(f"{status:<{MAX_MESSAGE_WIDTH}} {check_name:<{MAX_TITLE_WIDTH}}: {count:,} dữ liệu")
    if count > 0:
        all_valid = False   

In [7]:
print("\n[4] KIỂM TRA NO FINDING")
print("-" * 80)

labels_series = dataset_df[LABEL_COLUMN].fillna("").astype(str).str.strip()
invalid_no_finding = (
    labels_series
    .str.split(LABEL_SEPARATOR, regex=False)
    .apply(lambda labels: NO_FINDING_LABEL in labels and len(labels) > 1)
    .sum()
)

print(f"{'[INFO]':<{MAX_MESSAGE_WIDTH}} {'No Finding':<{MAX_TITLE_WIDTH}}: {(labels_series == NO_FINDING_LABEL).sum():,} dữ liệu")
status = "[OK]" if invalid_no_finding == 0 else "[INVALID]"
print(f"{status  :<{MAX_MESSAGE_WIDTH}} {'No Finding + bệnh':<{MAX_TITLE_WIDTH}}: {int(invalid_no_finding):,} dữ liệu")

if invalid_no_finding > 0:
    all_valid = False

In [8]:
print("\n[5] KIỂM TRA TẬP HUẤN LUYỆN VÀ TẬP KIỂM TRA")
print("-" * 80)


with open(TRAIN_SPLIT_DIRECTORY, "r", encoding="utf-8") as file:
    train_val_images = [line.strip() for line in file if line.strip()]

with open(TEST_SPLIT_DIRECTORY, "r", encoding="utf-8") as file:
    test_images = [line.strip() for line in file if line.strip()]
    
dataset_images = set(dataset_df[ID_COLUMN].astype(str))
train_val_set = set(train_val_images)
test_set = set(test_images)

split_checks = {
    "Trùng dữ liệu tập huấn luyện ": len(train_val_images) - len(train_val_set),
    "Trùng dữ liệu tập kiểm tra"   : len(test_images) - len(test_set),
    
    "Thiếu dữ liệu tập huấn luyện": len(train_val_set - dataset_images),
    "Thiếu dữ liệu tập kiểm tra"  : len(test_set - dataset_images),
    
    "Trùng dữ liệu giữa hai tập"  : len(train_val_set & test_set),
}

for check_name, count in split_checks.items():
    status = "[OK]" if count == 0 else "[INVALID]"
    print(f"{status:<{MAX_MESSAGE_WIDTH}} {check_name:<{MAX_TITLE_WIDTH}}: {count:,} dũ liệu")
    if count > 0:
        all_valid = False

In [9]:
print("\n[6] CÁC LỚP BỆNH TRONG TẬP DỮ LIỆU")
print("-" * 80)

CLASSES = sorted({
    label.strip()
    for value in labels_series
    for label in value.split(LABEL_SEPARATOR)
    if label.strip() and label.strip() != NO_FINDING_LABEL
})

print(f"{'[INFO]':<{MAX_MESSAGE_WIDTH}} {'Số bệnh':<{MAX_TITLE_WIDTH}}: {len(CLASSES)} bệnh\n")
for index, label in enumerate(CLASSES, start=1):
    print(f"{index:>2}. {label}")
        

In [10]:
print("\n[7] KIỂM TRA ẢNH TRONG TẬP DỮ LIỆU")
print("-" * 80)

image_files = [
    path for path in IMAGE_DIRECTORY.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]

image_map = {path.name: path.resolve() for path in image_files}
required_split_images = train_val_set | test_set
missing_physical_images = required_split_images - set(image_map)

print(f"{'[INFO]':<{MAX_MESSAGE_WIDTH}} {'Ảnh tìm thấy':<{MAX_TITLE_WIDTH}}: {len(image_map):,}")
print(f"{'[INFO]':<{MAX_MESSAGE_WIDTH}} {'Ảnh khớp trong tập dữ liệu':<{MAX_TITLE_WIDTH}}: {len(required_split_images):,}")

if missing_physical_images:
    print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {'Ảnh bị thiếu':<{MAX_TITLE_WIDTH}}: {len(missing_physical_images):,}")
    print("Ví dụ:", sorted(missing_physical_images)[:10])
    all_valid = False
else:
    print(f"{'[OK]':<{MAX_MESSAGE_WIDTH}} {'Ảnh bị thiếu':<{MAX_TITLE_WIDTH}}: 0")

In [11]:
print("\n[8] KIỂM TRA ẢNH TRONG TẬP DỮ LIỆU")
print("-" * 80)

if not OUTPUT_DIRECTORY.exists():
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    
    print(f"{'[CREATED]':<{MAX_MESSAGE_WIDTH}} {'OUTPUT_DIRECTORY': <25}: {OUTPUT_DIRECTORY}")
    
else:
    
    print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'OUTPUT_DIRECTORY': <25}: {OUTPUT_DIRECTORY}")

# =========================
# Kiểm tra dữ liệu
# =========================

In [12]:
print("\n[9] KIỂM TRA CẤU HÌNH KHÁC")
print("-" * 80)

# NUMBER_OF_FOLDS
if isinstance(NUMBER_OF_FOLDS, int) and NUMBER_OF_FOLDS >= 2:
    print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'NUMBER_OF_FOLDS':<{MAX_TITLE_WIDTH}}: {NUMBER_OF_FOLDS}")
else:
    print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {'NUMBER_OF_FOLDS':<{MAX_TITLE_WIDTH}}: phải >= 1")
    all_valid = False

# RANDOM_SEED
if isinstance(RANDOM_SEED, int):
    print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'RANDOM_SEED':<{MAX_TITLE_WIDTH}}: {RANDOM_SEED}")
else:
    print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {'RANDOM_SEED':<{MAX_TITLE_WIDTH}}: phải là số")
    all_valid = False

# RESIZE_IMAGES
if isinstance(RESIZE_IMAGES, bool):
    
    print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'RESIZE_IMAGES':<{MAX_TITLE_WIDTH}}: {RESIZE_IMAGES}")
    
else:
    
    print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} RESIZE_IMAGES phải là bool (True/False), nhưng nhận được: {RESIZE_IMAGES}")
    all_valid = False

# IMAGE_SIZE
if isinstance(IMAGE_SIZE, int) and not isinstance(IMAGE_SIZE, bool):

    if IMAGE_SIZE > 0:
        
        print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'IMAGE_SIZE':<{MAX_TITLE_WIDTH}}: {IMAGE_SIZE} x {IMAGE_SIZE}")
        
    else:
        
        print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} IMAGE_SIZE phải lớn hơn 0, nhưng nhận được: {IMAGE_SIZE}")
        all_valid = False

else:
    
    print(f"{'[INVALID]'    :<{MAX_MESSAGE_WIDTH}} IMAGE_SIZE phải là số nguyên, nhưng nhận được: {IMAGE_SIZE}")
    all_valid = False

# Hiển thị cấu hình resize thực tế
if (
    isinstance(RESIZE_IMAGES, bool)
    and RESIZE_IMAGES
    and isinstance(IMAGE_SIZE, int)
    and not isinstance(IMAGE_SIZE, bool)
    and IMAGE_SIZE > 0
):
    RESIZED_DIRECTORY = OUTPUT_DIRECTORY / f"images"

    if not RESIZED_DIRECTORY.exists():
        RESIZED_DIRECTORY.mkdir(parents=True, exist_ok=True)
        
        print(f"{'[CREATED]':<{MAX_MESSAGE_WIDTH}} {'RESIZED_DIRECTORY':<{MAX_TITLE_WIDTH}}: {RESIZED_DIRECTORY}")
        
    else:
        
        print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'RESIZED_DIRECTORY':<{MAX_TITLE_WIDTH}}: {RESIZED_DIRECTORY}")

elif isinstance(RESIZE_IMAGES, bool) and not RESIZE_IMAGES:
    
    print(f"{'[INFO]'      :<{MAX_MESSAGE_WIDTH}} Resize ảnh đang tắt. Ảnh sẽ giữ kích thước gốc.")

In [13]:
print("\n" + "=" * 80)

if all_valid:
    
    print("TẤT CẢ DỮ LIỆU VÀ CẤU HÌNH ĐỀU HỢP LỆ.")
    
else:
    
    print("PHÁT HIỆN DỮ LIỆU HOẶC CẤU HÌNH KHÔNG HỢP LỆ.")
    print("KIỂM TRA CÁC MỤC [MISSING] / [INVALID] Ở TRÊN.")

    print("=" * 80)

    assert all_valid, "Thiết lập không hợp lệ."

In [14]:
train_df = dataset_df[dataset_df[ID_COLUMN].isin(train_val_set)].copy()
test_df = dataset_df[dataset_df[ID_COLUMN].isin(test_set)].copy()

# Sắp xếp theo danh sách
train_order = {name: index for index, name in enumerate(train_val_images)}
test_order = {name: index for index, name in enumerate(test_images)}

train_df["__order"] = train_df[ID_COLUMN].map(train_order)
test_df["__order"] = test_df[ID_COLUMN].map(test_order)

train_df = train_df.sort_values("__order").drop(columns="__order").reset_index(drop=True)
test_df = test_df.sort_values("__order").drop(columns="__order").reset_index(drop=True)

# Thống kê
total_images = len(train_df) + len(test_df)
total_patients = pd.concat([train_df[PATIENT_ID_COLUMN],test_df[PATIENT_ID_COLUMN]]).nunique()

print(f"{'Tập dữ liệu huấn luyện':<{MAX_TITLE_WIDTH}}: {len(train_df)   :<8,} ảnh | {train_df[PATIENT_ID_COLUMN].nunique()  :<8,} bệnh nhân")
print(f"{'Tập dữ liệu kiểm tra'  :<{MAX_TITLE_WIDTH}}: {len(test_df)    :<8,} ảnh | {test_df[PATIENT_ID_COLUMN].nunique()       :<8,} bệnh nhân")
print(f"{'Tổng cộng'             :<{MAX_TITLE_WIDTH}}: {total_images    :<8,} ảnh | {total_images                               :<8,} bệnh nhân")

In [15]:
def encode_labels(df, classes):
    df = df.copy()
    parsed_labels = df[LABEL_COLUMN].astype(str).str.split(LABEL_SEPARATOR)

    for label in classes:
        df[label] = parsed_labels.apply(lambda labels: int(label in labels)).astype(np.uint8)

    df["num_positive_labels"] = df[classes].sum(axis=1).astype(np.uint8)
    df["is_no_finding"] = df[LABEL_COLUMN].eq(NO_FINDING_LABEL).astype(np.uint8)

    # Kiểm tra No Finding phải có vector toàn 0
    invalid_no_finding_vector = ((df["is_no_finding"] == 1) & (df["num_positive_labels"] != 0)).sum()
    assert invalid_no_finding_vector == 0, "No Finding đang có nhãn bệnh dương tính."

    return df

train_df = encode_labels(train_df, CLASSES)
test_df = encode_labels(test_df, CLASSES)

print(train_df[[ID_COLUMN, PATIENT_ID_COLUMN, LABEL_COLUMN] + CLASSES].head())

In [16]:
# Tạo ma trận nhãn ở mức patient
patient_labels_df = (train_df.groupby(PATIENT_ID_COLUMN, as_index=False)[CLASSES].max())

patient_ids = patient_labels_df[PATIENT_ID_COLUMN].to_numpy()
patient_y = patient_labels_df[CLASSES].to_numpy(dtype=np.uint8)
patient_x = np.arange(len(patient_labels_df)).reshape(-1, 1)

print(f"Số patient dùng cho K-Fold: {len(patient_labels_df):<8,} bệnh nhân")
print(f"Số lớp                    : {patient_y.shape[1]    :<8,} lớp")


# Chia K-Folds theo bệnh nhân
mskf = MultilabelStratifiedKFold(
    n_splits=NUMBER_OF_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

patient_to_fold = {}

for fold_index, (_, val_patient_idx) in enumerate(
    mskf.split(patient_x, patient_y)
):
    fold_number = fold_index + 1

    for patient_id in patient_ids[val_patient_idx]:
        patient_to_fold[patient_id] = fold_number

assert len(patient_to_fold) == len(patient_labels_df), "Có patient chưa được gán fold."


# Gán fold trở lại từng ảnh
train_df["Fold"] = (train_df[PATIENT_ID_COLUMN].map(patient_to_fold).astype(np.uint8))


# Kiểm tra
fold_counts = train_df["Fold"].value_counts().sort_index()

for fold, count in fold_counts.items():
    print(f"Fold {fold}: {count:,} ảnh")

In [17]:
expected_folds = set(range(1, NUMBER_OF_FOLDS + 1))
actual_folds = set(train_df["Fold"].dropna().astype(int).unique())

assert actual_folds == expected_folds, f"Fold thực tế {actual_folds} khác {expected_folds}."
assert train_df["Fold"].isna().sum() == 0, "Có ảnh chưa được gán fold."

In [18]:
fold_summary = []

for fold in range(1, NUMBER_OF_FOLDS + 1):

    fold_train = train_df[train_df["Fold"] != fold]
    fold_val   = train_df[train_df["Fold"] == fold]

    train_patients = set(fold_train[PATIENT_ID_COLUMN])
    val_patients   = set(fold_val[PATIENT_ID_COLUMN])

    patient_overlap = train_patients & val_patients

    assert len(patient_overlap) == 0, \
        f"Fold {fold} bị patient leakage."

    fold_summary.append({
        ("", "Fold"): fold,
        ("Số ảnh", "Huấn luyện"): len(fold_train),
        ("Số ảnh", "Kiểm thử"): len(fold_val),
        ("Số bệnh nhân", "Huấn luyện"): len(train_patients),
        ("Số bệnh nhân", "Kiểm thử"): len(val_patients),
    })

# Tạo DataFrame
fold_summary_df = pd.DataFrame(fold_summary)

# Chuyển tên cột thành MultiIndex để nhóm cột
fold_summary_df.columns = pd.MultiIndex.from_tuples(
    fold_summary_df.columns
)

# Định dạng bảng
styled_fold_summary = (
    fold_summary_df
    .style
    .hide(axis="index")
    .format({
        ("", "Fold"): "{:.0f}",
        ("Số ảnh", "Huấn luyện"): "{:,}",
        ("Số ảnh", "Kiểm thử"): "{:,}",
        ("Số bệnh nhân", "Huấn luyện"): "{:,}",
        ("Số bệnh nhân", "Kiểm thử"): "{:,}",
    })
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 15px",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-weight", "bold"),
                ("padding", "8px 15px"),
                ("border", "1px solid #cccccc"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("border", "1px solid #dddddd"),
            ],
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("margin", "auto"),
            ],
        },
    ])
)

display(styled_fold_summary)

In [19]:
# Thống kê số liệu
train_distribution_rows = []

for fold in range(1, NUMBER_OF_FOLDS + 1):

    fold_train = train_df[train_df["Fold"] != fold]
    train_total = len(fold_train)

    # 14 nhãn bệnh
    for label in CLASSES:

        train_positive = int(fold_train[label].sum())
        train_distribution_rows.append({
            "Lớp": label,
            "Fold": fold,
            "Số ảnh dương tính": train_positive,
            "Tổng số ảnh": train_total,
            "Tỷ lệ": (train_positive / train_total if train_total > 0 else 0.0),
        })

    # No Finding
    train_no_finding = int((fold_train[CLASSES].sum(axis=1) == 0).sum())
    train_distribution_rows.append({
        "Lớp": NO_FINDING_LABEL,
        "Fold": fold,
        "Số ảnh dương tính": train_no_finding,
        "Tổng số ảnh": train_total,
        "Tỷ lệ": (train_no_finding / train_total if train_total > 0 else 0.0),
    })

train_distribution_df = pd.DataFrame(train_distribution_rows)
label_order = [NO_FINDING_LABEL] + sorted(CLASSES)
train_distribution_df["Lớp"] = pd.Categorical(

train_distribution_df["Lớp"],
    categories=label_order,
    ordered=True
)

train_distribution_df = (
    train_distribution_df
    .sort_values(["Lớp", "Fold"])
    .reset_index(drop=True)
)


train_distribution_pivot = train_distribution_df.pivot(
    index="Lớp",
    columns="Fold",
    values="Tỷ lệ",
)

train_distribution_pivot = (train_distribution_pivot.reindex(label_order))

train_distribution_pivot.columns = [
    f"Fold {fold}"
    for fold in train_distribution_pivot.columns
]

# Bảng thống kê
train_distribution_table = (train_distribution_pivot.reset_index().copy())

train_distribution_table["Lớp"] = (
    train_distribution_table["Lớp"]
    .astype(str)
    .replace({"No Finding": "Không bệnh"})
)

display(
    train_distribution_table
    .style
    .hide(axis="index")
    .format({
        f"Fold {fold}": "{:.2%}"
        for fold in range(1, NUMBER_OF_FOLDS + 1)
    })
    .set_caption("Phân bố tỷ lệ các nhãn trong tập huấn luyện theo từng phần")
    .set_properties(**{
        "text-align": "center",
        "padding": "7px 12px",
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("margin-bottom", "10px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("vertical-align", "middle"),
                ("font-weight", "bold"),
                ("padding", "7px 12px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("vertical-align", "middle"),
            ],
        },
    ])
)

# Biểu đồ nhiệt
train_data = train_distribution_pivot.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(
    train_data,
    aspect="auto",
    cmap="Blues",
    vmin=0,
    vmax=train_data.max()
)

ax.set_xticks(
    np.arange(len(train_distribution_pivot.columns))
)

ax.set_xticklabels(
    train_distribution_pivot.columns,
    fontsize=10
)

train_display_labels = [
    "Không bệnh" if str(label) == "No Finding" else str(label)
    for label in train_distribution_pivot.index
]

ax.set_yticks(
    np.arange(len(train_distribution_pivot.index))
)

ax.set_yticklabels(
    train_display_labels,
    fontsize=9
)

threshold = train_data.max() * 0.55

for row in range(train_data.shape[0]):
    for col in range(train_data.shape[1]):

        value = train_data[row, col]

        ax.text(
            col,
            row,
            f"{value * 100:.2f}%",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if value >= threshold else "black"
        )

ax.set_title(
    "Phân bố tỷ lệ các nhãn trong tập huấn luyện theo từng phần",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Fold", fontsize=12)
ax.set_ylabel("Nhãn bệnh", fontsize=12)

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Tỷ lệ (%)", fontsize=12)
cbar.ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.set_xticks(np.arange(-0.5, train_data.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, train_data.shape[0], 1), minor=True)
ax.grid(which="minor", linewidth=0.5, alpha=0.25)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()
plt.show()

In [20]:
# Thống kê số liệu
validation_distribution_rows = []

for fold in range(1, NUMBER_OF_FOLDS + 1):

    fold_val = train_df[train_df["Fold"] == fold]
    val_total = len(fold_val)

    # 14 nhãn bệnh
    for label in CLASSES:

        val_positive = int(fold_val[label].sum())

        validation_distribution_rows.append({
            "Lớp": label,
            "Fold": fold,
            "Số ảnh dương tính": val_positive,
            "Tổng số ảnh": val_total,
            "Tỷ lệ": (val_positive / val_total if val_total > 0 else 0.0),
        })

    # No Finding
    val_no_finding = int((fold_val[CLASSES].sum(axis=1) == 0).sum())

    validation_distribution_rows.append({
        "Lớp": NO_FINDING_LABEL,
        "Fold": fold,
        "Số ảnh dương tính": val_no_finding,
        "Tổng số ảnh": val_total,
        "Tỷ lệ": (val_no_finding / val_total if val_total > 0 else 0.0),
    })


validation_distribution_df = pd.DataFrame(validation_distribution_rows)

label_order = [NO_FINDING_LABEL] + sorted(CLASSES)

validation_distribution_df["Lớp"] = pd.Categorical(
    validation_distribution_df["Lớp"],
    categories=label_order,
    ordered=True
)

validation_distribution_df = (
    validation_distribution_df
    .sort_values(["Lớp", "Fold"])
    .reset_index(drop=True)
)


validation_distribution_pivot = validation_distribution_df.pivot(
    index="Lớp",
    columns="Fold",
    values="Tỷ lệ",
)

validation_distribution_pivot = (
    validation_distribution_pivot.reindex(label_order)
)

validation_distribution_pivot.columns = [
    f"Fold {fold}"
    for fold in validation_distribution_pivot.columns
]


# Bảng thống kê
validation_distribution_table = (validation_distribution_pivot.reset_index().copy())

validation_distribution_table["Lớp"] = (
    validation_distribution_table["Lớp"]
    .astype(str)
    .replace({"No Finding": "Không bệnh"})
)

display(
    validation_distribution_table
    .style
    .hide(axis="index")
    .format({
        f"Fold {fold}": "{:.2%}"
        for fold in range(1, NUMBER_OF_FOLDS + 1)
    })
    .set_caption("Phân bố tỷ lệ các nhãn trong tập kiểm thử theo từng phần")
    .set_properties(**{
        "text-align": "center",
        "padding": "7px 12px",
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("margin-bottom", "10px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("vertical-align", "middle"),
                ("font-weight", "bold"),
                ("padding", "7px 12px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("vertical-align", "middle"),
            ],
        },
    ])
)


# Biểu đồ nhiệt
validation_data = validation_distribution_pivot.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(
    validation_data,
    aspect="auto",
    cmap="Blues",
    vmin=0,
    vmax=validation_data.max()
)

ax.set_xticks(np.arange(len(validation_distribution_pivot.columns)))
ax.set_xticklabels(validation_distribution_pivot.columns, fontsize=10)

validation_display_labels = [
    "Không bệnh" if str(label) == "No Finding" else str(label)
    for label in validation_distribution_pivot.index
]

ax.set_yticks(np.arange(len(validation_distribution_pivot.index)))
ax.set_yticklabels(validation_display_labels, fontsize=9)
threshold = validation_data.max() * 0.55

for row in range(validation_data.shape[0]):
    for col in range(validation_data.shape[1]):
        value = validation_data[row, col]
        ax.text(
            col,
            row,
            f"{value * 100:.2f}%",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if value >= threshold else "black"
        )

ax.set_title(
    "Phân bố tỷ lệ các nhãn trong tập kiểm thử theo từng phần",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Fold", fontsize=12)
ax.set_ylabel("Nhãn bệnh", fontsize=12)
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Tỷ lệ (%)", fontsize=12)

cbar.ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.set_xticks(np.arange(-0.5, validation_data.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, validation_data.shape[0], 1), minor=True)
ax.grid(which="minor", linewidth=0.5, alpha=0.25)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()
plt.show()

In [21]:
def attach_image_path(df):
    df = df.copy()
    df["original_path"] = df[ID_COLUMN].map(
        lambda image_name: str(image_map[image_name]) if image_name in image_map else None
    )
    return df

train_df = attach_image_path(train_df)
test_df = attach_image_path(test_df)

for name, df in {
    "Tập dữ liệu huấn luyện": train_df,
    "Tập dữ liệu kiểm tra": test_df,
}.items():
    missing = int(df["original_path"].isna().sum())
    
    print(f"{name:<{MAX_TITLE_WIDTH}}: {'Ảnh không tìm thấy':<{MAX_MESSAGE_WIDTH}}: {missing:<8,} ảnh")
    assert missing == 0, f"{name} không tìm thấy trên ổ đĩa."

In [22]:
RESIZED_DIRECTORY.mkdir(parents=True, exist_ok=True)
all_df = pd.concat([train_df, test_df],ignore_index=True)

unique_images = (all_df[[ID_COLUMN, "original_path"]].drop_duplicates(subset=[ID_COLUMN]).reset_index(drop=True))
resized_paths = {}

for _, row in TqdmComma(
    unique_images.iterrows(),
    total=len(unique_images),
    desc="Điều chỉnh kích cỡ ảnh"
):

    image_name = row[ID_COLUMN]
    source_path = Path(row["original_path"])
    target_path = RESIZED_DIRECTORY / image_name

    # Không xử lý lại nếu ảnh đã tồn tại
    if not target_path.exists():

        with Image.open(source_path) as image:

            image = image.convert("L")

            image = image.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.LANCZOS
            )

            image.save(
                target_path,
                format="PNG"
            )

    resized_paths[image_name] = str(
        target_path.resolve()
    )

for df in [train_df, test_df]:

    df["image_path"] = (
        df[ID_COLUMN]
        .map(resized_paths)
    )

In [23]:

# =========================
# Khai báo thư viện
# =========================
from pathlib import Path
from matplotlib.ticker import PercentFormatter
import json
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# try:
#     from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
# except ImportError:
#     subprocess.check_call([
#         sys.executable,
#         "-m",
#         "pip",
#         "install",
#         "iterative-stratification",
#         "-q",
#     ])
#     from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
# =========================
# Đường dẫn cố định
# =========================

ROOT_DIRECTORY   = (Path.cwd().parent).resolve()
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff",}

from tqdm import tqdm

class TqdmComma(tqdm):

    @staticmethod
    def format_meter(n, total, elapsed, **kwargs):

        text = tqdm.format_meter(
            n=n,
            total=total,
            elapsed=elapsed,
            **kwargs
        )

        if total is not None:

            old_value = f"{int(n)}/{int(total)}"
            new_value = f"{int(n):,}/{int(total):,}"

            text = text.replace(
                old_value,
                new_value
            )

        return text

In [24]:
RESIZED_DIRECTORY.mkdir(parents=True, exist_ok=True)
all_df = pd.concat([train_df, test_df],ignore_index=True)

unique_images = (all_df[[ID_COLUMN, "original_path"]].drop_duplicates(subset=[ID_COLUMN]).reset_index(drop=True))
resized_paths = {}

for _, row in TqdmComma(
    unique_images.iterrows(),
    total=len(unique_images),
    desc="Điều chỉnh kích cỡ ảnh"
):

    image_name = row[ID_COLUMN]
    source_path = Path(row["original_path"])
    target_path = RESIZED_DIRECTORY / image_name

    # Không xử lý lại nếu ảnh đã tồn tại
    if not target_path.exists():

        with Image.open(source_path) as image:

            image = image.convert("L")

            image = image.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.LANCZOS
            )

            image.save(
                target_path,
                format="PNG"
            )

    resized_paths[image_name] = str(
        target_path.resolve()
    )

for df in [train_df, test_df]:

    df["image_path"] = (
        df[ID_COLUMN]
        .map(resized_paths)
    )

In [25]:

# =========================
# Khai báo thư viện
# =========================
from pathlib import Path
from matplotlib.ticker import PercentFormatter
import json
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import shutil
# try:
#     from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
# except ImportError:
#     subprocess.check_call([
#         sys.executable,
#         "-m",
#         "pip",
#         "install",
#         "iterative-stratification",
#         "-q",
#     ])
#     from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
# =========================
# Đường dẫn cố định
# =========================

ROOT_DIRECTORY   = (Path.cwd().parent).resolve()
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff",}

from tqdm import tqdm

class TqdmComma(tqdm):

    @staticmethod
    def format_meter(n, total, elapsed, **kwargs):

        text = tqdm.format_meter(
            n=n,
            total=total,
            elapsed=elapsed,
            **kwargs
        )

        if total is not None:

            old_value = f"{int(n)}/{int(total)}"
            new_value = f"{int(n):,}/{int(total):,}"

            text = text.replace(
                old_value,
                new_value
            )

        return text

In [26]:
if RESIZED_DIRECTORY.exists():
    shutil.rmtree(RESIZED_DIRECTORY)

RESIZED_DIRECTORY.mkdir(parents=True, exist_ok=True)
all_df = pd.concat([train_df, test_df],ignore_index=True)

unique_images = (all_df[[ID_COLUMN, "original_path"]].drop_duplicates(subset=[ID_COLUMN]).reset_index(drop=True))
resized_paths = {}

for _, row in TqdmComma(
    unique_images.iterrows(),
    total=len(unique_images),
    desc="Điều chỉnh kích cỡ ảnh"
):

    image_name = row[ID_COLUMN]
    source_path = Path(row["original_path"])
    target_path = RESIZED_DIRECTORY / image_name

    # Không xử lý lại nếu ảnh đã tồn tại
    if not target_path.exists():

        with Image.open(source_path) as image:

            image = image.convert("L")

            image = image.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.LANCZOS
            )

            image.save(
                target_path,
                format="PNG"
            )

    resized_paths[image_name] = str(
        target_path.resolve()
    )

for df in [train_df, test_df]:

    df["image_path"] = (
        df[ID_COLUMN]
        .map(resized_paths)
    )

In [27]:
def calculate_class_statistics(df, classes):
    rows = []

    for label in classes:
        positive = int(df[label].sum())
        negative = int(len(df) - positive)
        positive_rate = positive / len(df) if len(df) else 0.0
        pos_weight = negative / positive if positive > 0 else 1.0

        rows.append({
            "class": label,
            "positive": positive,
            "negative": negative,
            "positive_rate": positive_rate,
            "pos_weight": pos_weight,
        })

    return pd.DataFrame(rows)

fold_outputs = {}

for fold in range(1, NUMBER_OF_FOLDS + 1):
    fold_dir = OUTPUT_DIRECTORY / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    fold_train_df = train_df[train_df["Fold"] != fold].copy().reset_index(drop=True)
    fold_validation_df = train_df[train_df["Fold"] == fold].copy().reset_index(drop=True)

    # Safety check
    overlap = (
        set(fold_train_df[PATIENT_ID_COLUMN])
        & set(fold_validation_df[PATIENT_ID_COLUMN])
    )
    assert len(overlap) == 0, f"Fold {fold}: patient leakage."

    class_statistics_df = calculate_class_statistics(fold_train_df, CLASSES)
    pos_weight = class_statistics_df["pos_weight"].to_numpy(dtype=np.float32)

    fold_train_df.to_csv(fold_dir / "train.csv", index=False)
    fold_validation_df.to_csv(fold_dir / "validation.csv", index=False)
    class_statistics_df.to_csv(fold_dir / "class_statistics.csv", index=False)
    np.save(fold_dir / "pos_weight.npy", pos_weight)

    fold_outputs[fold] = {
        "train_images": len(fold_train_df),
        "validation_images": len(fold_validation_df),
        "train_patients": fold_train_df[PATIENT_ID_COLUMN].nunique(),
        "validation_patients": fold_validation_df[PATIENT_ID_COLUMN].nunique(),
    }

pd.DataFrame.from_dict(fold_outputs, orient="index").rename_axis("fold")

In [28]:
def calculate_class_statistics(df, classes):
    rows = []

    for label in classes:
        positive = int(df[label].sum())
        negative = int(len(df) - positive)
        positive_rate = positive / len(df) if len(df) else 0.0
        pos_weight = negative / positive if positive > 0 else 1.0

        rows.append({
            "class": label,
            "positive": positive,
            "negative": negative,
            "positive_rate": positive_rate,
            "pos_weight": pos_weight,
        })

    return pd.DataFrame(rows)

fold_outputs = {}

for fold in range(1, NUMBER_OF_FOLDS + 1):
    fold_directory = OUTPUT_DIRECTORY / "folds" / f"fold_{fold}"
    fold_directory.mkdir(parents=True, exist_ok=True)

    fold_train_df = train_df[train_df["Fold"] != fold].copy().reset_index(drop=True)
    fold_validation_df = train_df[train_df["Fold"] == fold].copy().reset_index(drop=True)

    # Safety check
    overlap = (
        set(fold_train_df[PATIENT_ID_COLUMN])
        & set(fold_validation_df[PATIENT_ID_COLUMN])
    )
    assert len(overlap) == 0, f"Fold {fold}: patient leakage."

    class_statistics_df = calculate_class_statistics(fold_train_df, CLASSES)
    pos_weight = class_statistics_df["pos_weight"].to_numpy(dtype=np.float32)

    fold_train_df.to_csv(fold_directory / "train.csv", index=False)
    fold_validation_df.to_csv(fold_directory / "validation.csv", index=False)
    class_statistics_df.to_csv(fold_directory / "class_statistics.csv", index=False)
    np.save(fold_directory / "pos_weight.npy", pos_weight)

In [29]:
def calculate_class_statistics(df, classes):
    rows = []

    for label in classes:
        positive = int(df[label].sum())
        negative = int(len(df) - positive)
        positive_rate = positive / len(df) if len(df) else 0.0
        pos_weight = negative / positive if positive > 0 else 1.0

        rows.append({
            "class": label,
            "positive": positive,
            "negative": negative,
            "positive_rate": positive_rate,
            "pos_weight": pos_weight,
        })

    return pd.DataFrame(rows)

fold_outputs = {}

for fold in range(1, NUMBER_OF_FOLDS + 1):
    fold_directory = OUTPUT_DIRECTORY / "folds" / f"fold_{fold}"
    fold_directory.mkdir(parents=True, exist_ok=True)

    fold_train_df = train_df[train_df["Fold"] != fold].copy().reset_index(drop=True)
    fold_validation_df = train_df[train_df["Fold"] == fold].copy().reset_index(drop=True)

    # Safety check
    overlap = (
        set(fold_train_df[PATIENT_ID_COLUMN])
        & set(fold_validation_df[PATIENT_ID_COLUMN])
    )
    assert len(overlap) == 0, f"Fold {fold}: patient leakage."

    class_statistics_df = calculate_class_statistics(fold_train_df, CLASSES)
    pos_weight = class_statistics_df["pos_weight"].to_numpy(dtype=np.float32)

    fold_train_df.to_csv(fold_directory / "train.csv", index=False)
    fold_validation_df.to_csv(fold_directory / "validation.csv", index=False)
    class_statistics_df.to_csv(fold_directory / "class_statistics.csv", index=False)
    np.save(fold_directory / "pos_weight.npy", pos_weight)
    
print("Đã lưu thành công")

In [30]:
# Dữ liệu tổng có cột fold
train_df.to_csv(OUTPUT_DIRECTORY / "train_folds.csv", index=False)

# Official test được giữ nguyên, không có fold
if "Fold" in test_df.columns:
    test_df = test_df.drop(columns=["Fold"])
test_df.to_csv(OUTPUT_DIRECTORY / "test.csv", index=False)

# Thống kê fold
fold_distribution_rows = []

for fold in range(1, NUMBER_OF_FOLDS + 1):

    fold_val = train_df[train_df["Fold"] == fold]
    total = len(fold_val)

    # 14 bệnh
    for label in CLASSES:

        positive = int(fold_val[label].sum())

        fold_distribution_rows.append({
            "fold": fold,
            "class": label,
            "positive_images": positive,
            "total_images": total,
            "positive_rate": (
                positive / total
                if total > 0 else 0.0
            ),
        })

    # No Finding
    no_finding = int(
        (fold_val[CLASSES].sum(axis=1) == 0).sum()
    )

    fold_distribution_rows.append({
        "fold": fold,
        "class": NO_FINDING_LABEL,
        "positive_images": no_finding,
        "total_images": total,
        "positive_rate": (
            no_finding / total
            if total > 0 else 0.0
        ),
    })

fold_distribution_df = pd.DataFrame(
    fold_distribution_rows
)

fold_summary_df.to_csv(OUTPUT_DIRECTORY / "fold_summary.csv", index=False)

fold_distribution_df.to_csv(OUTPUT_DIRECTORY / "fold_distribution.csv", index=False)

# Thông tin classes + preprocessing
with open(OUTPUT_DIRECTORY / "classes.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "num_classes": len(CLASSES),
            "classes": CLASSES,
            "no_finding": NO_FINDING_LABEL,
            "no_finding_is_class": False,
            "label_separator": LABEL_SEPARATOR,
            "num_folds": NUMBER_OF_FOLDS,
            "random_state": RANDOM_SEED,
            "split_strategy": "patient-level multilabel stratified k-fold",
            "official_test_preserved": True,
            "image_size": IMAGE_SIZE if RESIZE_IMAGES else None,
        },
        file,
        indent=4,
        ensure_ascii=False,
    )

print("ĐÃ LƯU THÀNH CÔNG")
print("-" * 100)
print("Train/Val + folds :", OUTPUT_DIRECTORY / "train_val_folds.csv")
print("Official test     :", OUTPUT_DIRECTORY / "test.csv")
print("Fold directory    :", OUTPUT_DIRECTORY / "folds")
print("Classes metadata  :", OUTPUT_DIRECTORY / "classes.json")

In [31]:
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)

# Các file tổng
required_output_files = [
    OUTPUT_DIRECTORY / "train_val_folds.csv",
    OUTPUT_DIRECTORY / "test.csv",
    OUTPUT_DIRECTORY / "fold_summary.csv",
    OUTPUT_DIRECTORY / "fold_distribution.csv",
    OUTPUT_DIRECTORY / "classes.json",
]

for path in required_output_files:
    assert path.exists(), f"Thiếu output: {path}"
    print(f"[OK] {path.name}")

# File từng fold
for fold in range(1, N_SPLITS + 1):
    fold_dir = FOLDS_DIRECTORY / f"fold_{fold}"
    for filename in ["train.csv", "validation.csv", "class_statistics.csv", "pos_weight.npy"]:
        path = fold_dir / filename
        assert path.exists(), f"Thiếu output: {path}"

    saved_train = pd.read_csv(fold_dir / "train.csv")
    saved_val = pd.read_csv(fold_dir / "validation.csv")

    patient_overlap = (
        set(saved_train[PATIENT_ID_COLUMN])
        & set(saved_val[PATIENT_ID_COLUMN])
    )
    assert len(patient_overlap) == 0, f"Fold {fold} bị leakage sau khi lưu."

    print(
        f"[OK] Fold {fold}: "
        f"train={len(saved_train):,}, "
        f"validation={len(saved_val):,}, "
        f"patient_overlap=0"
    )

saved_test = pd.read_csv(OUTPUT_DIRECTORY / "test.csv")
assert len(saved_test) == len(test_df), "Số lượng test thay đổi sau khi lưu."

print("=" * 100)
print("PIPELINE HOÀN TẤT VÀ KHÔNG PHÁT HIỆN PATIENT LEAKAGE.")
print("=" * 100)

In [32]:
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)

# Các file tổng
required_output_files = [
    OUTPUT_DIRECTORY / "train_folds.csv",
    OUTPUT_DIRECTORY / "test.csv",
    OUTPUT_DIRECTORY / "fold_summary.csv",
    OUTPUT_DIRECTORY / "fold_distribution.csv",
    OUTPUT_DIRECTORY / "classes.json",
]

for path in required_output_files:
    assert path.exists(), f"Thiếu output: {path}"
    print(f"[OK] {path.name}")

# File từng fold
for fold in range(1, N_SPLITS + 1):
    fold_dir = FOLDS_DIRECTORY / f"fold_{fold}"
    for filename in ["train.csv", "validation.csv", "class_statistics.csv", "pos_weight.npy"]:
        path = fold_dir / filename
        assert path.exists(), f"Thiếu output: {path}"

    saved_train = pd.read_csv(fold_dir / "train.csv")
    saved_val = pd.read_csv(fold_dir / "validation.csv")

    patient_overlap = (
        set(saved_train[PATIENT_ID_COLUMN])
        & set(saved_val[PATIENT_ID_COLUMN])
    )
    assert len(patient_overlap) == 0, f"Fold {fold} bị leakage sau khi lưu."

    print(
        f"[OK] Fold {fold}: "
        f"train={len(saved_train):,}, "
        f"validation={len(saved_val):,}, "
        f"patient_overlap=0"
    )

saved_test = pd.read_csv(OUTPUT_DIRECTORY / "test.csv")
assert len(saved_test) == len(test_df), "Số lượng test thay đổi sau khi lưu."

print("=" * 100)
print("PIPELINE HOÀN TẤT VÀ KHÔNG PHÁT HIỆN PATIENT LEAKAGE.")
print("=" * 100)

In [33]:
# Dữ liệu tổng có cột fold
train_df.to_csv(OUTPUT_DIRECTORY / "train_folds.csv", index=False)

# Official test được giữ nguyên, không có fold
if "Fold" in test_df.columns:
    test_df = test_df.drop(columns=["Fold"])
test_df.to_csv(OUTPUT_DIRECTORY / "test.csv", index=False)


# Thông tin classes + preprocessing
with open(OUTPUT_DIRECTORY / "classes.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "num_classes": len(CLASSES),
            "classes": CLASSES,
            "no_finding": NO_FINDING_LABEL,
            "no_finding_is_class": False,
            "label_separator": LABEL_SEPARATOR,
            "number_of_folds": NUMBER_OF_FOLDS,
            "random_seed": RANDOM_SEED,
            "split_strategy": "patient-level multilabel stratified k-fold",
            "official_test_preserved": True,
            "image_size": IMAGE_SIZE if RESIZE_IMAGES else None,
        },
        file,
        indent=4,
        ensure_ascii=False,
    )

print("ĐÃ TIỀN XỬ LÝ DỮ LIỆU THÀNH CÔNG")
print("-" * 100)

In [34]:
# Dữ liệu tổng có cột fold
train_df.to_csv(OUTPUT_DIRECTORY / "train_folds.csv", index=False)

# Official test được giữ nguyên, không có fold
if "Fold" in test_df.columns:
    test_df = test_df.drop(columns=["Fold"])
test_df.to_csv(OUTPUT_DIRECTORY / "test.csv", index=False)


# Thông tin classes + preprocessing
with open(OUTPUT_DIRECTORY / "classes.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "num_classes": len(CLASSES),
            "classes": CLASSES,
            "no_finding": NO_FINDING_LABEL,
            "no_finding_is_class": False,
            "label_separator": LABEL_SEPARATOR,
            "number_of_folds": NUMBER_OF_FOLDS,
            "random_seed": RANDOM_SEED,
            "split_strategy": "patient-level multilabel stratified k-fold",
            "official_test_preserved": True,
            "image_size": IMAGE_SIZE if RESIZE_IMAGES else None,
        },
        file,
        indent=4,
        ensure_ascii=False,
    )

print("ĐÃ TIỀN XỬ LÝ DỮ LIỆU THÀNH CÔNG")
print("-" * 100)